# 🎯 [OPS04_M02] 다방향 동시 침투 항적 데이터 정제 및 센서별 경보 메시지 큐잉 전처리 구현
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS04/ops04_m02_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS04_M02 |
| **미션 주제** | 다방향 동시 침투 항적 데이터 정제 및 센서별 경보 메시지 큐잉 전처리 구현 |
| **소요 시간** | 약 2.0시간 |
| **핵심 산출물** | `src/preprocess/ops04_preprocessor.py, 전처리 전후 데이터셋` |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

## 📂 2. 표준 산출물 디렉토리 체계 및 실습 팁
- **방법 A (`%%writefile` 저장 추천):** 코드 셀 첫 줄에 `%%writefile <산출물 경로>`(예: `%%writefile src/models/ops04_baseline_model.py`)를 추가해 해당 폴더에 모듈을 저장할 수 있습니다.
- **방법 B (셀 단위 논리 대응 인정):** 별도 파일 추출 없이 노트북 셀 단위로 독립 실행·검증하고 미션 10 통합을 완료해도 정상 인정됩니다.

> **⚠️ [교육용 가상 데이터셋 및 HITL 원칙]**
> - 본 노트북의 모든 좌표·비행체 유형·수치는 보안 규정을 준수하여 **교육용 가상 데이터로 생성**됩니다.
> - 실제 무기 제원·사거리·부대 위치·군사 기밀 데이터는 일체 사용하지 않습니다.
> - 산출되는 대응 순위는 지휘관 승인을 위한 **'권고'**이며 자동 승인·자동 실행은 없습니다.

---

In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 다방향 동시 침투 항적 데이터 정제 및 센서별 경보 메시지 큐잉 전처리 구현
**소요시간**: 약 2.0시간 | **이번 산출물**: `src/preprocess/ops04_preprocessor.py, 전처리 전후 데이터셋`

### 📋 미션 개요
**다방향·다중 센서에서 동시에 유입되는 경보 메시지를 시각 순으로 정렬하고, 중복·반사파·스푸핑 잡음을 걸러내는 실시간 전처리 파이프라인을 구축합니다.**

- **배경 및 필요성:** 포화 공격 시 다수 센서가 동일 표적을 중복 보고하거나 반사파·스푸핑·센서 단절이 뒤섞여 들어오므로, 정제되지 않은 원시 데이터를 그대로 우선순위 엔진에 넘기면 상황판 과부하와 오경보가 발생합니다.
- **핵심 과업:** 센서별 큐 정렬 및 k-way 병합으로 타임스탬프 순서를 맞추고, 재전송 중복 및 약한 신호 반사파 제거, 1초 창 센서 간 보고 융합, 속도 대비 위치 도약 스푸핑 항적 격리 및 센서 단절 구간 표시를 구현합니다.
- **학습 목표 (습득 역량):** 다중 스트림 시계열 병합(k-way merge) 구현 능력, 전술 센서 노이즈(반사파·스푸핑·단절) 판별 및 정제 로직 설계 역량, 전처리 성능·오류율 정량 검증 능력을 습득합니다.
- **검증 및 산출물:** 처리 속도 1,000건/초 이상, 전처리 오류율 0%(정렬 위반 0건)를 달성한 전처리 모듈(`src/preprocess/ops04_preprocessor.py`)과 전처리 전후 데이터셋을 산출합니다.

### 🎓 학습 목표 (습득 역량)
**[습득 역량]** 다방향·다중 센서에서 비동기로 유입되는 경보 스트림을 시간순으로 병합·정규화하고, 전술 현장의 중복·반사파·스푸핑 잡음을 정제하는 실시간 데이터 전처리 파이프라인 구축 역량을 체득합니다.

**[핵심 과업]** 센서별 큐 정렬 및 k-way 병합으로 타임스탬프 순서를 맞추고, 재전송 중복·반사파 유령 항적 제거, 1초 창 센서 간 융합, 위치 도약 스푸핑 항적 격리 및 센서 단절 구간 표시 후 처리 속도와 오류율을 측정합니다.

- **다중 스트림 시계열 정렬(k-way 병합) 구현 역량:** 센서별 수신 지연과 순서 뒤섞임이 발생해도 타임스탬프 기준 전역 순서를 무결하게 복원하는 큐잉·병합 알고리즘 설계 능력을 익힙니다.
- **전술 센서 노이즈 정제 및 다중 보고 융합 능력:** 동일 센서의 재전송 중복, 약한 신호 반사파 유령 항적, 센서 단절 구간을 구분해 제거·표시하고 1초 창 내 동일 항적 보고를 하나로 융합하는 기술을 습득합니다.
- **운동학적 스푸핑 격리 및 성능 계측 역량:** 보고 속도에 비해 위치가 비정상적으로 도약하는 스푸핑 항적을 격리하고, 초당 처리량(1,000건/초 이상)과 정답 라벨 대비 오류율(0%)을 정량 검증하는 능력을 기릅니다.

### 📖 미션 배경 및 시나리오
#### 미션 배경 및 당위성
다방향에서 동시 유입되는 대량의 경보 메시지를 타임스탬프 순으로 정제하고 센서별 중복 메시지를 큐잉하는 실시간 데이터 전처리 파이프라인을 구축한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **여러 방향·여러 센서에서 동시에 들어오는 경보 메시지를 시각 순으로 정렬하고 중복·잡음을 걸러내는 전처리 파이프라인을 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **센서별 큐에 쌓인 메시지를 각각 정렬한 뒤 k-way 병합으로 타임스탬프 순서를 맞춘다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/preprocess/ops04_preprocessor.py 및 전처리 전후 데이터셋` 형태의 결과물을 완성한다.

### 🤖 단계별 수행 및 AI 협업 가이드
#### Step 1. 미션 2 코드를 실행해 전처리 전 메시지 유형별 건수를 확인한다.
#### 이 단계의 역할
미션 2의 `make_stream()`과 `preprocess()` 파이프라인을 실행해 전처리 전후 메시지 건수와 정제·융합 통계를 확인합니다.

#### 해야 할 일

- 4개 가상 센서(`SENSORS`)에서 생성된 원시 메시지 스트림(8,100건)의 유형별 분포(정상·중복·반사파·스푸핑·단절)를 확인한다.
- AI를 Co-pilot으로 활용해 센서별 큐 정렬(`deque`) 및 `heapq` 기반 k-way 병합 로직의 시간 정렬 원리를 검토한다.
- `preprocess()` 실행 후 융합 보고 건수(3,530건)와 정렬 위반 0건 여부를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "4개 센서 큐에 쌓인 경보 메시지를 Python heapq를 이용해 타임스탬프 순으로 k-way 병합하고, 1초 시간 창 내 동일 항적 보고를 융합하는 preprocess() 함수 구조를 설명하고 최적화 포인트를 짚어줘."

  - **AI 결과 검증 포인트:** AI가 제안한 k-way 병합 로직에서 동일 타임스탬프 충돌 시 정렬 키 오류(dict 비교 TypeError 등)가 발생하지 않도록 `(ts, sensor_idx, seq, msg)` 형태의 튜플 키를 쓰는지 확인합니다.

#### 핵심 확인 지표

> 메시지 8,100건 -> 융합 보고 3,530건, 오류율 0.00%, 정렬 위반 0건

#### 학습자가 제출해야 하는 내용

- 미션 2 코드 실행 결과 및 전처리 전후 건수 비교 로그
- AI 프롬프트 질의 및 k-way 병합 정렬 키 검증 기록

#### Step 2. 중복·반사파·스푸핑·센서 단절 처리 건수를 기록한다.
#### 이 단계의 역할
중복 재전송, 약한 신호 반사파, 운동학적 스푸핑, 센서 단절 구간의 정제 건수를 채점용 정답 라벨과 대조 검증합니다.

#### 해야 할 일

- 동일 센서 재전송 중복, 낮은 SNR의 반사파 유령 항적, 속도 대비 위치 도약 스푸핑 항적, 센서 단절 구간의 판별 조건을 점검한다.
- AI에게 유형별 필터링 카운트(중복 612건, 반사파 103건, 스푸핑 항적 6개 등)와 누락/오분류 케이스를 점검하는 집계 스크립트를 요청한다.
- 각 노이즈 유형의 제거·격리 건수와 정답 라벨 대비 정합성을 표로 기록한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "전처리 결과에서 재전송 중복 제거 건수, SNR 임계값 미만 반사파 제거 건수, 속도 대비 위치 도약 스푸핑 격리 항적 수, 센서 단절 표시 건수를 정답 라벨과 비교해 정밀도/오류율을 출력하는 검증 코드를 작성해줘."

  - **AI 결과 검증 포인트:** AI가 반사파 제거 임계값이나 위치 도약 기준을 임의로 변경해 정상 고속 표적(fast_type)을 스푸핑으로 오분류하지 않는지 유형별 오분류 표로 검증합니다.

#### 핵심 확인 지표

> 중복 612건 제거, 반사파 103건 제거, 스푸핑 항적 6개 격리, 오류율 0.00%

#### 학습자가 제출해야 하는 내용

- 유형별 정제·격리 건수 및 정답 라벨 대조표
- AI 생성 검증 코드 및 오분류 점검 기록

#### Step 3. 처리 속도와 오류율이 목표를 넘는지 확인한다.
#### 이 단계의 역할
대량 경보 스트림 처리 속도가 목표(1,000건/초 이상)를 넘고 전처리 오류율이 0%인지 성능 지표를 검증합니다.

#### 해야 할 일

- `time.perf_counter()` 기반 처리 시간 계측 결과를 확인해 초당 처리 건수(건/초)를 산출한다.
- AI에게 `preprocess()` 내 병목 구간(예: 불필요한 리스트 순회나 반복 정렬)을 진단하고 처리량을 높이는 리팩토링 방안을 질의한다.
- 리팩토링 전후 처리 속도와 정렬 위반 0건·오류율 0% 유지 여부를 검증한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "8,100건 센서 메시지를 처리하는 preprocess() 함수의 초당 처리량(건/초)과 타임스탬프 역전 여부를 측정하고, O(N log K) 병합 성능을 유지하면서 병목을 줄이는 개선안을 제시해줘."

  - **AI 결과 검증 포인트:** 속도 최적화를 위해 AI가 중복 제거용 시간 창(1초)이나 센서 단절 판정 로직을 생략해 오류율이 0%에서 깨지지 않았는지 반드시 재테스트합니다.

#### 핵심 확인 지표

> 데이터 정제 처리 속도 1,000건/초 이상, 전처리 오류율 0.00%, 정렬 위반 0건

#### 학습자가 제출해야 하는 내용

- 처리 속도(건/초) 및 오류율 측정 로그
- AI 성능 개선 제안 검토 및 회귀 검증 메모

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
`preprocess()`의 `snr_min` 임계값을 `0.25`에서 `0.45`로 상향 조정했을 때 정상 메시지 오삭제 발생 여부와 원인을 분석합니다.

#### 해야 할 일

- `snr_min` 파라미터를 `0.25`에서 `0.45`로 변경해 `preprocess()`를 재실행한다.
- AI에게 SNR 임계값 상향이 반사파 제거율과 정상 저신호 항적 보존(재현율)에 미치는 트레이드오프 분석을 요청한다.
- 변경 전후 융합 보고 건수와 정상 메시지 오삭제 건수를 비교해 최적 임계값 선정 근거를 정리한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "preprocess()에서 반사파 제거 기준인 snr_min을 0.25에서 0.45로 올렸을 때, 반사파 제거 건수와 정상 표적 오삭제 건수가 어떻게 변하는지 비교표로 출력하고 전술적 트레이드오프를 설명해줘."

  - **AI 결과 검증 포인트:** AI의 설명이 실제 시뮬레이션 출력 수치(오삭제 건수 증가 여부)와 일치하는지 확인하고, 악천후·원거리 탐지 시 저신호 정상 드론이 누락될 위험을 논리적으로 점검합니다.

#### 핵심 확인 지표

> snr_min 0.25 vs 0.45 변경 전후 오삭제·반사파 제거 비교 및 최적 근거 도출

#### 학습자가 제출해야 하는 내용

- snr_min 변경 전후 비교 분석 메모 및 최적 임계값 근거
- AI 질의 프롬프트 및 트레이드오프 검증 기록

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가영역:** 전술 환경 특유의 노이즈(스푸핑, 반사파, 센서 단절 등) 제거 타당성

#### 미션 PASS 조건
- 정량 기준: 데이터 정제 처리 속도 1,000건/초 이상, 전처리 오류율 0%
- 진행 기준: 정제 파이프라인 자동화 및 단위 테스트 케이스 5종 통과
- 검증 방법: pytest 기반 전처리 정합성 테스트 및 결측치 보정율 확인
- AI 검증 기준: k-way 병합 정렬 키 및 중복·반사파·스푸핑 정제 로직을 AI와 협업 구현하고 정답 라벨 대비 오류율 0.00% 및 snr_min 트레이드오프 교차 검증 완료
#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 AI 프롬프트 설계의 구체성, AI 산출물의 오류·보안 위반을 인간 검토(HITL)로 식별·수정한 기록, 그리고 출력 수치의 군 실무 부합성을 종합 채점한다.

In [ ]:
# [TODO : 학생 실습 2] 핵심 함수의 TODO 영역을 AI Co-pilot과 함께 구현하고 검증하세요.
"""다방향에서 동시에 들어오는 가상 센서 경보 메시지를 센서별 큐에 받아 타임스탬프 순으로 정렬하고, 중복·반사파(유령 항적)·스푸핑·센서 단절을 정제하는 전처리 파이프라인을 구현해 처리 속도와 오류율을 측정한다."""
import heapq
import time
from collections import defaultdict, deque
import numpy as np
import pandas as pd

# ---------------- [교육용가상자료] 가상 방어지역·가상 비행체 생성기 ----------------
# 좌표는 가상 지휘소를 원점으로 한 km 단위 평면이다. 비행체 유형은 추상 범주이며
# 속도·고도·반사강도는 학습용 임의 범위로, 실제 무기 제원·사거리·탄도와 무관하다.
ASSETS = pd.DataFrame([["A1", "CP(virtual)", 0.0, 0.0, 1.0],
                       ["A2", "Radar(virtual)", 4.0, -3.0, 0.8],
                       ["A3", "Depot(virtual)", -5.0, 4.0, 0.7],
                       ["A4", "Power(virtual)", 7.0, 6.0, 0.5]], columns=["asset", "name", "x", "y", "value"])
TYPES = {  # 유형: (위협 여부, 속도 m/s 범위, 고도 m 범위, 출현 비율)
    "uav_type": (True, (30, 60), (100, 800), 0.42),      # 저속 무인기형(가상)
    "fast_type": (True, (150, 250), (50, 200), 0.16),    # 고속 순항형(가상)
    "decoy": (False, (30, 60), (100, 800), 0.16),        # 기만체(가상)
    "civil": (False, (120, 220), (3000, 9000), 0.10),    # 민항기(가상)
    "bird": (False, (8, 18), (50, 400), 0.16),           # 조류 군집
}
TTI_URGENT = 600      # 긴급 판정 기준: 가상 자산 도달 예상시간 600초 이내
CPA_KM = 1.5          # 자산 상공 통과 판정 반경(가상)


def make_raid(seed, n=80, spoof=0):
    """가상 포화 상황 1회분(비행체 n개 + 스푸핑 가짜 항적 spoof개)을 만든다."""
    rng = np.random.default_rng(seed)
    names = list(TYPES)
    rows = []
    for i in range(n + spoof):
        typ = "spoof" if i >= n else str(rng.choice(names, p=[TYPES[k][3] for k in names]))
        threat, sp, al, _ = TYPES["decoy" if typ == "spoof" else typ]
        brg = np.radians(rng.uniform(0, 360)); dist = rng.uniform(6, 60)
        x, y = dist * np.sin(brg), dist * np.cos(brg)
        spd = rng.uniform(*sp); alt = rng.uniform(*al)
        aim = int(rng.integers(0, len(ASSETS)))
        if typ in ("uav_type", "fast_type", "decoy", "spoof"):      # 가상 자산 방향(작은 방위 오차)
            ax_, ay_ = ASSETS.loc[aim, ["x", "y"]]
            hd = np.arctan2(ax_ - x, ay_ - y) + np.radians(rng.normal(0, 0.6))
        else:
            hd = rng.uniform(0, 2 * np.pi); aim = -1
        vx, vy = spd * np.sin(hd) / 1000, spd * np.cos(hd) / 1000          # km/s
        sig = {"uav_type": ((0.2, 0.5), (0.02, 0.12), [0, .1, .5, .4]), "fast_type": ((0.1, 0.4), (0.02, 0.12), [0, .2, .4, .4]),
               "decoy": ((0.3, 0.8), (0.08, 0.35), [0, .55, .4, .05]), "spoof": ((0.3, 0.8), (0.05, 0.3), [0, .7, .3, 0]),
               "civil": ((0.7, 1.0), (0.01, 0.05), [0, 0, .1, .9]), "bird": ((0.01, 0.12), (0.1, 0.4), [0, .6, .4, 0])}[typ]
        jump = spd * rng.uniform(0.95, 1.2) if typ != "spoof" else spd * rng.uniform(4, 15)
        true_tti = np.inf
        if threat:
            d_aim = np.hypot(ASSETS.loc[aim, "x"] - x, ASSETS.loc[aim, "y"] - y)
            true_tti = d_aim * 1000 / spd
        rows.append({"id": f"T{seed % 100:02d}{i:03d}", "type": typ, "x": x, "y": y, "vx": vx, "vy": vy, "alt": alt, "spd": spd,
                     "rcs": rng.uniform(*sig[0]), "rcs_var": rng.uniform(*sig[1]), "n_sensors": int(rng.choice(4, p=sig[2])),
                     "max_jump": jump, "civ_iff": int(typ == "civil" and rng.random() < 0.95),
                     "aim": aim, "threat": threat, "true_tti": true_tti,
                     "urgent": bool(threat and true_tti <= TTI_URGENT)})
    df = pd.DataFrame(rows)
    df["true_prio"] = [ASSETS.loc[a, "value"] * np.exp(-t / 300) if th else 0.0
                       for a, t, th in zip(df["aim"], df["true_tti"], df["threat"])]
    return df


def geometry(df):
    """각 항적과 가상 자산 사이 최근접점(CPA) 거리와 도달 예상시간(TTI, s)을 계산한다."""
    px, py = df["x"].to_numpy()[:, None], df["y"].to_numpy()[:, None]
    vx, vy = df["vx"].to_numpy()[:, None], df["vy"].to_numpy()[:, None]
    rx, ry = ASSETS["x"].to_numpy()[None] - px, ASSETS["y"].to_numpy()[None] - py
    v2 = vx ** 2 + vy ** 2 + 1e-12
    t = (rx * vx + ry * vy) / v2                                  # CPA 도달 시각(s)
    cpa = np.hypot(rx - vx * t, ry - vy * t)
    tti = np.where((t > 0) & (cpa < CPA_KM), t, np.inf)
    return cpa, tti

SENSORS = ["RDR-1", "RDR-2", "EO-1", "ACU-1"]


def make_stream(seed=21, n=120, steps=30):
    """[교육용가상자료] 센서별 메시지 스트림. label: valid/dup/ghost/spoof (채점용 정답)"""
    rng = np.random.default_rng(seed)
    raid = make_raid(seed, n, spoof=6)
    queues = {s: [] for s in SENSORS}
    mid = 0
    for r in raid.itertuples():
        sens = list(rng.choice(SENSORS, size=max(1, r.n_sensors), replace=False))
        for k in range(steps):
            if r.Index % 9 == 0 and 10 <= k < 15:          # 일부 항적은 센서 단절 구간
                continue
            t = k * 1000 + int(rng.integers(0, 300))
            x, y = r.x + r.vx * k, r.y + r.vy * k
            if r.type == "spoof":                         # 스푸핑: 위치가 비정상적으로 도약
                x, y = x + rng.normal(0, 3), y + rng.normal(0, 3)
            for s in sens:
                m = {"mid": mid, "sensor": s, "track": r.id, "ts": t + int(rng.integers(0, 80)), "x": x, "y": y,
                     "spd": r.spd, "snr": rng.uniform(0.4, 1.0), "label": "spoof" if r.type == "spoof" else "valid"}
                queues[s].append(m); mid += 1
                if rng.random() < 0.08:                   # 중복 재전송
                    queues[s].append({**m, "mid": mid, "label": "dup"}); mid += 1
                if s.startswith("RDR") and rng.random() < 0.03:   # 반사파 유령 항적(약한 신호, 대칭 위치)
                    queues[s].append({**m, "mid": mid, "track": "G" + r.id[1:], "x": -x, "snr": rng.uniform(0.05, 0.2),
                                      "label": "ghost"}); mid += 1
    for s in SENSORS:                                      # 전송 지연으로 순서가 섞여 도착
        q = queues[s]
        idx = rng.permutation(len(q))[: len(q) // 10]
        for a, b in zip(idx[::2], idx[1::2]):
            q[a], q[b] = q[b], q[a]
    return queues


def preprocess(queues, snr_min=0.25, jump_ratio=3.0, gap_ms=5000):
    """[TODO : 학생 실습 2] 센서 큐 k-way 병합, 중복·반사파(snr_min) 제거, 1초 창 융합 및 스푸핑 격리를 구현하세요."""
    # 힌트: heapq.merge로 (ts, mid) 정렬 -> 중복/snr_min 필터링 -> (track, sec) 1초 창 융합 -> 속도 대비 도약(jump_ratio) 격리
    stats = defaultdict(int)
    fused, spoofed, gaps = {}, set(), 0
    # <<< 여기에 전처리 파이프라인 코드를 작성하세요 >>>
    return list(fused.values()), spoofed, gaps, stats


queues = make_stream()
n_in = sum(len(q) for q in queues.values())
t0 = time.perf_counter()
out, stats, merged = preprocess(queues)
dt = time.perf_counter() - t0
rate = n_in / dt

kept = {mid for f in out for mid in f["src"]}
labels = {m["mid"]: m["label"] for m in merged}
wrong_drop = sum(1 for k, l in labels.items() if l == "valid" and k not in kept)
wrong_keep = sum(1 for k, l in labels.items() if l != "valid" and k in kept)
err = (wrong_drop + wrong_keep) / n_in

print(f"[전처리 전] 메시지 {n_in}건 (센서 {len(queues)}개), 유형별:", pd.Series(labels).value_counts().to_dict())
print(f"[전처리 후] 융합 보고 {len(out)}건, 항적 {len({f['track'] for f in out})}개")
for k, v in stats.items():
    print(f"  - {k}: {v}")
ts = [m["ts"] for m in merged]
print(f"  - 타임스탬프 정렬 위반: {sum(b < a for a, b in zip(ts, ts[1:]))}건")
multi = sum(len(f["sensors"]) > 1 for f in out) / len(out)
print(f"  - 다중 센서 융합 비율: {multi:.1%}")
print(f"\n처리 시간 {dt * 1000:.1f} ms, 처리 속도 {rate:,.0f}건/초 (목표 1,000건/초 이상) -> {'달성' if rate >= 1000 else '미달'}")
print(f"정상 메시지 오삭제 {wrong_drop}건, 잡음 미제거 {wrong_keep}건, 전처리 오류율 {err:.2%} (목표 0%) -> {'달성' if err == 0 else '미달'}")
print("\n[전처리 후 데이터 앞부분]")
print(pd.DataFrame([{"track": f["track"], "sec": f["sec"], "x_km": round(f["x"], 2), "y_km": round(f["y"], 2), "센서수": len(f["sensors"])}
                    for f in sorted(out, key=lambda f: (f["sec"], f["track"]))[:6]]).to_string(index=False))

<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
[전처리 전] 메시지 8100건 (센서 4개), 유형별: {'valid': 7145, 'dup': 612, 'spoof': 240, 'ghost': 103}
[전처리 후] 융합 보고 3530건, 항적 120개
  - 반사파 제거: 103
  - 중복 제거: 612
  - 스푸핑 항적 격리: 6
  - 센서 단절 구간 표시: 14
  - 타임스탬프 정렬 위반: 0건
  - 다중 센서 융합 비율: 71.8%

처리 시간 49.3 ms, 처리 속도 164,466건/초 (목표 1,000건/초 이상) -> 달성
정상 메시지 오삭제 0건, 잡음 미제거 0건, 전처리 오류율 0.00% (목표 0%) -> 달성

[전처리 후 데이터 앞부분]
 track  sec   x_km   y_km  센서수
T21000    0 -27.36 -34.88    3
T21001    0  50.43  13.12    2
T21002    0   0.03  20.05    3
T21003    0  -8.61 -25.31    3
T21004    0 -13.38 -24.24    1
T21005    0  50.83 -22.68    2
```
</details>
